In [1]:
import pandas as pd
import numpy as np
import scipy as sp
import os
import dask
import dask.dataframe as dd
import itertools
from itertools import chain
from math import sqrt, floor, ceil, isnan
import multiprocess
import multiprocessing
import importlib
from importlib import reload
from collections import Counter
from fuzzywuzzy import process, fuzz
import time
import seaborn as sns
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import warnings
import pyreadstat
import bisect
warnings.filterwarnings("error")

pd.options.display.max_columns = 500
pd.options.display.max_rows = 1000
pd.options.display.max_colwidth = 400

# A customized winsorisation function that handles None values correctly
# The percentiles are taken and winsorisation are done on non-None values only
def winsor2(series,cutoffs):

    import numpy as np
    import scipy as sp
    
    IsNone = np.isnan(series).copy()
    IsNotNone = np.logical_not(IsNone).copy()
    series_NotNonePart = sp.stats.mstats.winsorize(series[IsNotNone],limits=(cutoffs[0],cutoffs[1]))
    series_new = series.copy()
    series_new[IsNone] = np.nan
    series_new[IsNotNone] = series_NotNonePart

    return series_new


In [10]:
try:
    del(FUN_0J_Process_Liquidity)
except:
    pass
import FUN_0J_Process_Liquidity
importlib.reload(FUN_0J_Process_Liquidity)
from FUN_0J_Process_Liquidity import FUN_0J_Process_Liquidity

input = [[item] for item in list(range(2005,2023))]
if __name__ == '__main__':
    with multiprocessing.Pool(processes = 4) as p:
        liquidity_allyears = p.starmap(FUN_0J_Process_Liquidity,input)
liquidity = pd.concat(liquidity_allyears)

In [89]:
# Supplement with bond-level info
GPF = pd.read_csv("../CleanData/SDC/0A_GPF.csv",low_memory=False)

# Maturity date and amount of the Thompson-researched version is used, which is complete in trading data sample period
# and is aligned with the number of CUSIPs
GPF = GPF[['Issuer','issuer_type','issuer_type_full','cusip',
    'dated_date','sale_date','sale_year','County','County_raw','State',
    'TOM_amount_by_maturity','TOM_maturity_date',
    'Bid','security_type','taxable_code',
    'CBSA Code','CSA Code','CBSA Title','CSA Title']].copy()

GPF['cusip_list'] = GPF['cusip'].str.split('\n')
GPF['TOM_amount_by_maturity_list'] = GPF['TOM_amount_by_maturity'].str.split('\n')
GPF['TOM_maturity_date_list'] = GPF['TOM_maturity_date'].str.split('\n')

GPF = GPF[~pd.isnull(GPF['cusip'])]
GPF = GPF[~pd.isnull(GPF['TOM_amount_by_maturity_list'])]
GPF = GPF[~pd.isnull(GPF['TOM_maturity_date_list'])]

In [70]:
def proc_list(GPF):
    GPF_bondlevel = []
    for idx,row in GPF.iterrows():
        if len(row['cusip_list'])!=len(row['TOM_amount_by_maturity_list']):
            continue
        if len(row['TOM_amount_by_maturity_list'])!=len(row['TOM_maturity_date_list']):
            continue
        for j in range(len(row['cusip_list'])):
            GPF_bondlevel = GPF_bondlevel+[{
                'Issuer':row['Issuer'],
                'issuer_type':row['issuer_type'],
                'issuer_type_full':row['issuer_type_full'],
                'dated_date':row['dated_date'],
                'sale_date':row['sale_date'],
                'sale_year':row['sale_year'],
                'County':row['County'],
                'County_raw':row['County_raw'],
                'State':row['State'],
                'Bid':row['Bid'],
                'security_type':row['security_type'],
                'taxable_code':row['taxable_code'],
                'CBSA Code':row['CBSA Code'],
                'CSA Code':row['CSA Code'],
                'CBSA Title':row['CBSA Title'],
                'CSA Title':row['CSA Title'],
                'CUSIP':row['cusip_list'][j],
                'TOM_amount':row['TOM_amount_by_maturity_list'][j],
                'TOM_maturity_date':row['TOM_maturity_date_list'][j],
                }]
    GPF_bondlevel = pd.DataFrame(GPF_bondlevel)
    return GPF_bondlevel

meta_columns = list(proc_list(GPF[:10]).columns)
GPF_dd = dd.from_pandas(GPF, npartitions=20)
with dask.config.set(scheduler='processes',num_workers=20):
    GPF = GPF_dd.map_partitions(proc_list,meta=pd.DataFrame(columns=meta_columns)).compute()

In [99]:
liquidity_issueinfo = liquidity.merge(GPF,on='CUSIP')

liquidity_issueinfo['TOM_amount'] = liquidity_issueinfo['TOM_amount'].astype(float)
liquidity_issueinfo['dollar_trades_scaled'] = \
    liquidity_issueinfo['dollar_trades']/liquidity_issueinfo['TOM_amount']/1_000_000
liquidity_issueinfo.to_parquet("../CleanData/SDC/0J_liquidity_issueinfo.parquet")